# GLN 数据处理教程

## 概述

本 notebook 展示 GLN（Graph Logic Network）的**完整数据处理流程**。

GLN 的数据处理流水线将原始反应数据逐步转化为模型可用的图特征格式，经历以下 6 个阶段：

```
原始反应 CSV
  ↓ Step 1: SMILES 标准化
标准化的分子集合
  ↓ Step 2: 反应模板提取
反应模板集（SMARTS 格式）
  ↓ Step 3: SMARTS 标准化 
标准化的反应中心/反应物中心
  ↓ Step 4: 反应中心匹配
产物 → 可匹配中心的映射表
  ↓ Step 5: 正负样本构建
正确模板 + 负采样模板/反应物
  ↓ Step 6: 图特征导出
二进制分子图文件（节点特征 + 边特征 + 拓扑）
```

### 教学方式

为了便于理解，本 notebook **不依赖 GLN 原始代码的 import**，而是用最小化的纯 Python + RDKit 代码重现每一步的核心逻辑。我们使用一个**微型数据集**（3 条反应）来完整演示整个流程。

---

In [1]:
# ====== 基础导入 ======
import numpy as np
import csv
import os
import pickle
from collections import defaultdict, Counter
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from IPython.display import display, HTML

print('所有导入成功！')

所有导入成功！


## 构建微型教学数据集

我们读取本教程已经配置好的demo数据。每条反应包含：
- **reactants**: 反应物
- **reagents**: 试剂/催化剂
- **production**: 产物
- **class**: 反应类型

数据格式与 Schneider50k 数据集一致：`reactants>reagents>production`。

In [2]:
# ====== 构造微型数据集 ======
# 每条记录：(反应类型, 反应SMILES)
# 反应SMILES格式: reactants>reagents>product (原子带映射编号)

import pandas as pd

demo_data = pd.read_csv('data/demo_data.csv')

MINI_REACTIONS = list(zip(demo_data['class'], demo_data['reactants>reagents>production']))

print(f'教学数据集包含 {len(MINI_REACTIONS)} 条反应：')
print('='*80)
for i, (rxn_type, rxn) in enumerate(MINI_REACTIONS[:5]):  # 只显示前5条反应
    reactants, reagents, product = rxn.split('>')
    print(f'\n反应 {i+1} (类型 {rxn_type}):')
    print(f'  反应物: {reactants}')
    print(f'  产物:   {product}')

教学数据集包含 1000 条反应：

反应 1 (类型 nan):
  反应物: C1CCOC1.CCOCC.Cc1ccccc1.O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[c:4]([n:3][c:2]([CH3:1])[s:6]2)[CH2:15][CH2:14]1.[NH:11]([CH3:12])[CH3:13]
  产物:   [CH3:1][c:2]1[n:3][c:4]2[c:5]([s:6]1)[C:7](=[O:8])/[C:9](=[CH:10]\[N:11]([CH3:12])[CH3:13])[CH2:14][CH2:15]2

反应 2 (类型 nan):
  反应物: CCN(CC)CC.CCOCC.CN(C)C=O.O=C1CCC(=O)N1O[C:2](=[O:1])[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1.[NH2:3][C@H:4]1[CH:5]=[CH:6][C@@H:7]([OH:8])[CH2:9][CH2:10]1
  产物:   [O:1]=[C:2]([NH:3][CH:4]1[CH:5]=[CH:6][CH:7]([OH:8])[CH2:9][CH2:10]1)[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1

反应 3 (类型 nan):
  反应物: Cc1c(CC(=O)Cl)[c:26]2[c:5](n1[C:11](=[O:12])[c:13]1[cH:14][cH:15][c:16](Cl)[cH:17][cH:18]1)[cH:4][cH:3][c:2](O[CH3:1])[cH:27]2.Cc1c(CC(=O)O)c2cc(O[CH3:23])ccc2n1C(=O)[c:19]1[cH:20][cH:21][c:22](Cl)[cH:24][cH:25]1.Cl.ClCCl.O=S(Cl)Cl.[nH:6]1[cH:7][cH:8][n:10][cH:9]1
  产物:   [CH3:1][c:2]1[cH:3][cH:4][c:5]([N:6]=[CH:7]/[CH:8]=[CH:9]/[N:10]([C:11](=[O:12])[c

---

## Step 1: SMILES 标准化

### 为什么需要标准化？

SMILES 表示不唯一——同一个分子可以有多种 SMILES 写法。例如：
- 苯: `c1ccccc1` 或 `C1=CC=CC=C1`
- 乙醇: `CCO` 或 `OCC` 或 `[CH3][CH2][OH]`

GLN 需要将所有 SMILES 统一为 **canonical（标准）** 形式，以保证：
1. 同一分子只有唯一的 SMILES 表示
2. 去除原子映射编号（atom map number），因为模型不需要
3. 去除显式氢原子，简化分子表示

### GLN 源码对应
- 文件: `gln/data_process/get_canonical_smiles.py`
- 函数: `gln/common/mol_utils.py → cano_smiles()`

In [3]:
# ====== Step 1: SMILES 标准化 ======
# 对应 GLN 源码: gln/data_process/get_canonical_smiles.py

def cano_smiles(smiles):
    """
    将 SMILES 标准化。
    处理步骤：
    1. 解析 SMILES → 分子对象
    2. 移除显式氢原子 (RemoveHs)
    3. 清除原子映射编号 (ClearProp('molAtomMapNumber'))
    4. 重新生成 canonical SMILES
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None, smiles
        mol = Chem.RemoveHs(mol)
        if mol is None:
            return None, smiles
        # 清除原子映射编号
        [a.ClearProp('molAtomMapNumber') for a in mol.GetAtoms()]
        return mol, Chem.MolToSmiles(mol)
    except:
        return None, smiles


def process_smiles(reactions):
    """
    处理所有反应中出现的 SMILES，构建标准化映射表。
    对应 GLN 源码: get_canonical_smiles.py → process_smiles()
    """
    smiles_cano_map = {}  # 原始SMILES → 标准化SMILES
    all_symbols = set()   # 收集所有出现的原子类型

    for rxn_type, rxn in reactions:
        reactants, _, prod = rxn.split('>')
        # 收集所有分子: 反应物 + 产物
        all_mols_smiles = reactants.split('.') + [prod]

        for sm in all_mols_smiles:
            mol, cano_sm = cano_smiles(sm)
            if mol is not None:
                for a in mol.GetAtoms():
                    all_symbols.add((a.GetAtomicNum(), a.GetSymbol()))
            smiles_cano_map[sm] = cano_sm

    return smiles_cano_map, sorted(all_symbols)


# 执行 Step 1
smiles_cano_map, atom_list = process_smiles(MINI_REACTIONS)

print(f'Step 1 完成: SMILES 标准化')
print(f'  总共处理 SMILES 数量: {len(smiles_cano_map)}')
unique_mols = set(smiles_cano_map.values())
print(f'  去重后唯一分子数量:   {len(unique_mols)}')
print(f'  出现的原子类型:       {atom_list}')
print()
print('标准化映射示例（前5项）:')
print('-' * 60)
for i, (orig, cano) in enumerate(smiles_cano_map.items()):
    if i >= 5:
        break
    print(f'  {orig:35s} → {cano}')

[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not removing hydrogen atom without neighbors
[11:11:54] WARNING: not r

Step 1 完成: SMILES 标准化
  总共处理 SMILES 数量: 3156
  去重后唯一分子数量:   2902
  出现的原子类型:       [(1, 'H'), (3, 'Li'), (5, 'B'), (6, 'C'), (7, 'N'), (8, 'O'), (9, 'F'), (11, 'Na'), (12, 'Mg'), (13, 'Al'), (14, 'Si'), (15, 'P'), (16, 'S'), (17, 'Cl'), (19, 'K'), (20, 'Ca'), (22, 'Ti'), (24, 'Cr'), (25, 'Mn'), (26, 'Fe'), (28, 'Ni'), (29, 'Cu'), (30, 'Zn'), (35, 'Br'), (43, 'Tc'), (45, 'Rh'), (46, 'Pd'), (47, 'Ag'), (50, 'Sn'), (53, 'I'), (55, 'Cs'), (58, 'Ce'), (76, 'Os'), (78, 'Pt'), (79, 'Au'), (82, 'Pb')]

标准化映射示例（前5项）:
------------------------------------------------------------
  C1CCOC1                             → C1CCOC1
  CCOCC                               → CCOCC
  Cc1ccccc1                           → Cc1ccccc1
  O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[c:4]([n:3][c:2]([CH3:1])[s:6]2)[CH2:15][CH2:14]1 → Cc1nc2c(s1)C(=O)C(C=O)CC2
  [NH:11]([CH3:12])[CH3:13]           → CNC


[11:11:55] WARNING: not removing hydrogen atom without neighbors
[11:11:55] WARNING: not removing hydrogen atom without neighbors
[11:11:55] WARNING: not removing hydrogen atom without neighbors
[11:11:55] WARNING: not removing hydrogen atom without neighbors


---

## Step 2: 反应模板提取

### 什么是反应模板？

反应模板（Reaction Template）用 **SMARTS** 格式描述一类反应的「规则」，它只保留反应前后发生变化的部分（**反应中心**），而忽略不变的部分。

模板格式: `[产物中心SMARTS]>>[反应物中心SMARTS]`

例如，酯化反应的模板可能是：
```
[C:1](=[O:2])[O:3][C:4]>>[C:1](=[O:2])[OH:3].[C:4][OH]
```
含义：产物中的酯键 C(=O)O-C → 断裂为羧酸 C(=O)OH + 醇 C-OH

### 提取方法

GLN 使用 **RDChiral** 的 `extract_from_reaction()` 来自动提取模板。这里我们用一个简化版本来演示核心逻辑。

### GLN 源码对应
- 文件: `gln/data_process/build_raw_template.py`
- 依赖: `gln/mods/rdchiral/template_extractor.py`

In [4]:
# ====== Step 2: 反应模板提取 ======
# 对应 GLN 源码: gln/data_process/build_raw_template.py

# 这里我们用 rdChiral 的模板提取器
# 如果 rdchiral 未安装，我们提供替代的简化实现

try:
    from rdchiral.template_extractor import extract_from_reaction
    USE_RDCHIRAL = True
    print('使用 rdChiral 进行模板提取')
except ImportError:
    USE_RDCHIRAL = False
    print('rdChiral 未安装，使用简化模板提取（手动定义）')


def extract_template(rxn_smiles, rxn_id='0'):
    """
    从反应SMILES中提取反应模板。
    
    GLN 的做法是用 rdChiral 将完整反应 SMILES 分解为：
    - 产物中心 SMARTS (product center)
    - 反应物中心 SMARTS (reactant center)
    组合为 retro_template: [product_center]>>[reactant_center]
    """
    reactants, reagent, product = rxn_smiles.split('>')
    
    if USE_RDCHIRAL:
        reaction = {
            '_id': rxn_id,
            'reactants': reactants,
            'products': product
        }
        result = extract_from_reaction(reaction)
        if 'reaction_smarts' in result:
            return result['reaction_smarts']
        else:
            return None
    else:
        # 简化版: 基于原子映射识别变化的原子来构建模板
        return _simple_template_extract(reactants, product)


def _simple_template_extract(reactants_smi, product_smi):
    """
    简化的模板提取：通过比较反应物和产物中原子的连接关系变化来识别反应中心。
    这是 rdChiral 核心逻辑的教学简化版。
    """
    # 解析产物
    prod_mol = Chem.MolFromSmiles(product_smi)
    if prod_mol is None:
        return None
    
    # 获取产物中的原子映射
    prod_map = {}
    for atom in prod_mol.GetAtoms():
        map_num = atom.GetAtomMapNum()
        if map_num > 0:
            prod_map[map_num] = atom.GetIdx()
    
    # 解析反应物
    react_mols = [Chem.MolFromSmiles(s) for s in reactants_smi.split('.')]
    react_map = {}
    for mol in react_mols:
        if mol is None:
            continue
        for atom in mol.GetAtoms():
            map_num = atom.GetAtomMapNum()
            if map_num > 0:
                react_map[map_num] = atom
    
    # 找出连接发生变化的原子 (反应中心)
    changed_maps = set()
    for map_num in prod_map:
        if map_num in react_map:
            prod_atom = prod_mol.GetAtomWithIdx(prod_map[map_num])
            react_atom = react_map[map_num]
            # 比较邻居连接
            prod_neighbors = set()
            for n in prod_atom.GetNeighbors():
                nm = n.GetAtomMapNum()
                if nm > 0:
                    prod_neighbors.add(nm)
            react_neighbors = set()
            for n in react_atom.GetNeighbors():
                nm = n.GetAtomMapNum()
                if nm > 0:
                    react_neighbors.add(nm)
            if prod_neighbors != react_neighbors:
                changed_maps.add(map_num)
    
    if len(changed_maps) == 0:
        return None
    
    # 简化输出：标记变化的原子
    return f'[反应中心原子映射: {sorted(changed_maps)}]'


# ====== 对训练集的每条反应提取模板 ======
extracted_templates = []

print('Step 2: 反应模板提取')
print('=' * 80)
for i, (rxn_type, rxn) in enumerate(MINI_REACTIONS):
    try:
        template = extract_template(rxn, rxn_id=str(i))
        extracted_templates.append({
            'id': i,
            'class': rxn_type,
            'rxn_smiles': rxn,
            'retro_template': template
        })
        print(f'\n反应 {i+1} (类型 {rxn_type}):')
        reactants, _, product = rxn.split('>')
        print(f'  反应: {reactants} → {product}')
        print(f'  提取模板: {template}')
    except Exception as e:
        print(f'  反应 {i+1} 模板提取失败: {e}')
        extracted_templates.append({
            'id': i,
            'class': rxn_type,
            'rxn_smiles': rxn,
            'retro_template': None
        })

# 统计
valid_templates = [t for t in extracted_templates if t['retro_template'] is not None]
print(f'\n\n成功提取模板: {len(valid_templates)}/{len(extracted_templates)}')

使用 rdChiral 进行模板提取
Step 2: 反应模板提取

反应 1 (类型 nan):
  反应: C1CCOC1.CCOCC.Cc1ccccc1.O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[c:4]([n:3][c:2]([CH3:1])[s:6]2)[CH2:15][CH2:14]1.[NH:11]([CH3:12])[CH3:13] → [CH3:1][c:2]1[n:3][c:4]2[c:5]([s:6]1)[C:7](=[O:8])/[C:9](=[CH:10]\[N:11]([CH3:12])[CH3:13])[CH2:14][CH2:15]2
  提取模板: [C:3]/[C;H0;D3;+0:2](=[CH;D2;+0:1]/[N;H0;D3;+0:7](-[C;D1;H3:6])-[C;D1;H3:8])-[C:4]=[O;D1;H0:5]>>O=[CH;D2;+0:1]-[CH;D3;+0:2](-[C:3])-[C:4]=[O;D1;H0:5].[C;D1;H3:6]-[NH;D2;+0:7]-[C;D1;H3:8]

反应 2 (类型 nan):
  反应: CCN(CC)CC.CCOCC.CN(C)C=O.O=C1CCC(=O)N1O[C:2](=[O:1])[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1.[NH2:3][C@H:4]1[CH:5]=[CH:6][C@@H:7]([OH:8])[CH2:9][CH2:10]1 → [O:1]=[C:2]([NH:3][CH:4]1[CH:5]=[CH:6][CH:7]([OH:8])[CH2:9][CH2:10]1)[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1
  提取模板: [#8:2]-[C;H0;D3;+0:1](=[O;D1;H0:3])-[NH;D2;+0:6]-[CH;D3;+0:5](-[C:4])-[C:7]=[C:8]>>O=C1-C-C-C(=O)-N-1-O-[C;H0;D3;+0:1](-[#8:2])=[O;D1;H0:3].[C:4]-[C@@H;D3;+0:5](-[NH2;D1;+0:

---

## Step 3: 模板过滤与 SMARTS 标准化

### 模板过滤

提取出的模板可能非常多。GLN 可以选择**过滤掉低频模板**（只出现 1-2 次的模板可能是噪声），但论文默认保留所有模板（`tpl_min_cnt=1`）。

### SMARTS 标准化

与 SMILES 类似，SMARTS 模式也需要标准化。GLN 将每个模板拆分为：
- **产物中心 SMARTS（prod center）**: 模板中描述产物结构的部分
- **反应物中心 SMARTS（react center）**: 模板中描述反应物结构的部分

并分别对它们进行标准化。

### GLN 源码对应
- 文件: `gln/data_process/filter_template.py`, `gln/data_process/get_canonical_smarts.py`

In [5]:
# ====== Step 3a: 模板过滤 ======
# 对应 GLN 源码: gln/data_process/filter_template.py

def filter_templates(templates_list, min_count=1):
    """
    过滤低频模板。
    
    参数:
        templates_list: 模板字典列表
        min_count: 最低出现次数阈值
    返回:
        过滤后的 (类型, 模板) 列表
    """
    # 统计模板出现次数
    tpl_counter = Counter()
    tpl_types = defaultdict(set)
    
    for item in templates_list:
        tpl = item['retro_template']
        if tpl is None:
            continue
        tpl_counter[tpl] += 1
        tpl_types[tpl].add(item['class'])
    
    print(f'总模板数: {len(tpl_counter)}')
    print(f'模板频次分布: {dict(tpl_counter)}')
    
    # 过滤
    filtered = []
    for tpl, count in tpl_counter.items():
        if count >= min_count:
            for rxn_type in tpl_types[tpl]:
                filtered.append((rxn_type, tpl))
    
    print(f'过滤后模板数 (min_count={min_count}): {len(filtered)}')
    return filtered

# 执行过滤（保留所有，min_count=1）
# 注意：GLN 论文中默认不过滤任何模板
filtered_templates = filter_templates(extracted_templates, min_count=1)

print('\n过滤后的模板列表:')
for rxn_type, tpl in filtered_templates:
    print(f'  类型 {rxn_type}: {tpl}')

总模板数: 814
模板频次分布: {'[C:3]/[C;H0;D3;+0:2](=[CH;D2;+0:1]/[N;H0;D3;+0:7](-[C;D1;H3:6])-[C;D1;H3:8])-[C:4]=[O;D1;H0:5]>>O=[CH;D2;+0:1]-[CH;D3;+0:2](-[C:3])-[C:4]=[O;D1;H0:5].[C;D1;H3:6]-[NH;D2;+0:7]-[C;D1;H3:8]': 1, '[#8:2]-[C;H0;D3;+0:1](=[O;D1;H0:3])-[NH;D2;+0:6]-[CH;D3;+0:5](-[C:4])-[C:7]=[C:8]>>O=C1-C-C-C(=O)-N-1-O-[C;H0;D3;+0:1](-[#8:2])=[O;D1;H0:3].[C:4]-[C@@H;D3;+0:5](-[NH2;D1;+0:6])-[C:7]=[C:8]': 1, '[CH3;D1;+0:4]-[c;H0;D3;+0:3]1:[c:2]:[cH;D2;+0:1]:[c;H0;D3;+0:7](-[N;H0;D2;+0:25]=[CH;D2;+0:21]/[CH;D2;+0:22]=[CH;D2;+0:24]/[N;H0;D3;+0:23](-[C;H0;D3;+0:8](=[O;D1;H0:9])-[c:10])-[c;H0;D3;+0:15]2:[c:16]:[c:17]:[c;H0;D3;+0:18](-[CH3;D1;+0:14]):[c:19]:[c:20]:2):[c:6]:[c:5]:1.[c:12]:[cH;D2;+0:11]:[c:13]>>C-c1:c(-C-C(-Cl)=O):[c;H0;D3;+0:1]2:[c:2]:[c;H0;D3;+0:3](-O-[CH3;D1;+0:4]):[c:5]:[c:6]:[c;H0;D3;+0:7]:2:n:1-[C;H0;D3;+0:8](=[O;D1;H0:9])-[c:10].Cl-[c;H0;D3;+0:11](:[c:12]):[c:13].C-c1:c(-C-C(-O)=O):c2:c:c(-O-[CH3;D1;+0:14]):c:c:c:2:n:1-C(=O)-[c;H0;D3;+0:15]1:[c:16]:[c:17]:[c;H0;D3;+0:18](-C

In [6]:
# ====== Step 3b: SMARTS 标准化 ======
# 对应 GLN 源码: gln/data_process/get_canonical_smarts.py

def cano_smarts(smarts):
    """
    标准化 SMARTS 模式。
    与 SMILES 标准化类似：去除原子映射、生成 canonical 表示。
    """
    mol = Chem.MolFromSmarts(smarts)
    if mol is None:
        return None, smarts
    [a.ClearProp('molAtomMapNumber') for a in mol.GetAtoms()]
    cano = Chem.MolToSmarts(mol)
    if '[[se]]' in cano:  # RDKit 已知的解析bug
        cano = smarts
    return mol, cano


def smarts_has_useless_parentheses(smarts):
    """检查 SMARTS 是否有多余的外层括号"""
    if len(smarts) == 0 or smarts[0] != '(' or smarts[-1] != ')':
        return False
    cnt = 1
    for i in range(1, len(smarts)):
        if smarts[i] == '(':
            cnt += 1
        if smarts[i] == ')':
            cnt -= 1
        if cnt == 0 and i + 1 != len(smarts):
            return False
    return True


def process_template_smarts(templates):
    """
    处理所有模板的 SMARTS：
    1. 拆分为产物中心和反应物中心
    2. 分别标准化
    3. 构建去重后的中心列表
    """
    prod_cano_smarts_set = set()  # 所有唯一的产物中心 SMARTS
    react_cano_smarts_set = set() # 所有唯一的反应物中心 SMARTS  
    smarts_cano_map = {}           # 原始 → 标准化映射
    
    for rxn_type, template in templates:
        # 模板格式: product_smarts>>reactant_smarts
        # 注意: GLN 的模板可能是 prod>agent>react 格式
        parts = template.split('>')
        if len(parts) == 3:
            sm_prod, _, sm_react = parts
        elif len(parts) == 2 and parts[0] == '':
            # >>react 格式
            continue
        else:
            # 跳过非标准格式
            continue
        
        # 处理产物中心
        if smarts_has_useless_parentheses(sm_prod):
            sm_prod = sm_prod[1:-1]
        _, cano_prod = cano_smarts(sm_prod)
        smarts_cano_map[sm_prod] = cano_prod
        prod_cano_smarts_set.add(cano_prod)
        
        # 处理反应物中心（可能有多个组分，用.分隔）
        for r_smarts in sm_react.split('.'):
            _, cano_react = cano_smarts(r_smarts)
            smarts_cano_map[r_smarts] = cano_react
            react_cano_smarts_set.add(cano_react)
    
    prod_cano_smarts = sorted(prod_cano_smarts_set)
    react_cano_smarts = sorted(react_cano_smarts_set)
    
    return prod_cano_smarts, react_cano_smarts, smarts_cano_map


# 执行 SMARTS 标准化
prod_cano_smarts, react_cano_smarts, smarts_cano_map = process_template_smarts(filtered_templates)

print(f'Step 3 完成: SMARTS 标准化')
print(f'  唯一产物中心数: {len(prod_cano_smarts)}')
print(f'  唯一反应物中心数: {len(react_cano_smarts)}')
print()
print('产物中心 SMARTS（用于子结构匹配）:')
for i, s in enumerate(prod_cano_smarts):
    print(f'  [{i}] {s}')
print()
print('反应物中心 SMARTS:')
for i, s in enumerate(react_cano_smarts):
    print(f'  [{i}] {s}')

Step 3 完成: SMARTS 标准化
  唯一产物中心数: 726
  唯一反应物中心数: 915

产物中心 SMARTS（用于子结构匹配）:
  [0] C#C-[C&H2&D2&+0]-[N&H0&D3&+0](-[C&D1&H3])-[C&D1&H3]
  [1] C#[C&H0&D2&+0]-[c&H0&D3&+0](:c):c
  [2] C#[C&H1&D1&+0]
  [3] C-[#14&H0&D4&+0](-[C&D1&H3])(-[C&D1&H3])-[O&H0&D2&+0]-C
  [4] C-[#14@@&H1&D3&+0](-[C&H2&D2&+0]-C)-C
  [5] C-[C&H0&D3&+0](-C(-[C&D1&H3])=[O&D1&H0])=[C&H0&D3&+0](-[C&D1&H3])-c:c:[c&H0&D3&+0](-[C&H3&D1&+0]):c
  [6] C-[C&H0&D3&+0](-C)=[O&H0&D1&+0]
  [7] C-[C&H0&D3&+0](-[C&D1&H3])=[O&H0&D1&+0]
  [8] C-[C&H0&D3&+0](-[N&H2&D1&+0])=[N&H1&D1&+0]
  [9] C-[C&H0&D3&+0](-[N&H2&D1&+0])=[O&D1&H0]
  [10] C-[C&H0&D3&+0](-[O&D1&H1])=[O&H0&D1&+0]
  [11] C-[C&H0&D3&+0](-[O&H1&D1&+0])=[C&H0&D3&+0](-C(-[N&D1&H2])=[O&D1&H0])-[C&H0&D3&+0](=[O&H0&D1&+0])-C-[C&H0&D3&+0](-[O&H1&D1&+0])=[C&H0&D3&+0](-C)-[C&H0&D3&+0](=[O&H0&D1&+0])-c
  [12] C-[C&H0&D3&+0](=[C&H1&D2&+0]-C=C)-C
  [13] C-[C&H0&D3&+0](=[N&H1&D1&+0])-[S&H0&D2&+0]-[C&H3&D1&+0]
  [14] C-[C&H0&D3&+0](=[O&D1&H0])-[N&H0&D3&+0](-C)-C
  [15] C-[C&H0&D3&+0](=[O&D

---

## Step 4: 反应中心匹配

### 核心思想

对于训练/验证/测试集中的每个产物分子，我们需要找出所有**可能匹配**的反应中心。这是通过 RDKit 的 **子结构匹配**（`HasSubstructMatch`）来实现的。

匹配逻辑：
```
对于每个产物 p:
  对于每个产物中心 SMARTS c:
    如果 p 包含子结构 c:
      记录 (产物p, 反应类型) → 中心c
```

这步生成的映射表是推理时的重要数据结构——给定一个新产物，我们可以快速查找哪些反应中心可能适用。

### GLN 源码对应
- 文件: `gln/data_process/find_centers.py`
- 函数: `find_edges()`

In [7]:
# ====== Step 4: 反应中心匹配 ======
# 对应 GLN 源码: gln/data_process/find_centers.py

def find_product_centers(reactions, smiles_cano_map, prod_cano_smarts, smarts_cano_map, filtered_templates):
    """
    对每个反应的产物，查找所有匹配的反应中心。
    
    返回: dict  (rxn_type, cano_product) → [center_idx1, center_idx2, ...]
    """
    # 预编译 SMARTS 分子对象
    center_mols = []
    for sm in prod_cano_smarts:
        center_mols.append((sm, Chem.MolFromSmarts(sm)))
    
    # 构建 SMARTS → 反应类型 的映射（只在正确的反应类型下匹配）
    smarts_type_set = defaultdict(set)
    for rxn_type, template in filtered_templates:
        parts = template.split('>')
        if len(parts) != 3:
            continue
        sm_prod = parts[0]
        if smarts_has_useless_parentheses(sm_prod):
            sm_prod = sm_prod[1:-1]
        if sm_prod in smarts_cano_map:
            cano = smarts_cano_map[sm_prod]
            smarts_type_set[cano].add(rxn_type)
    
    # 对每个产物做子结构匹配
    prod_center_maps = {}
    
    for rxn_type, rxn in reactions:
        _, _, raw_prod = rxn.split('>')
        cano_prod = smiles_cano_map.get(raw_prod, raw_prod)
        
        mol = Chem.MolFromSmiles(cano_prod)
        if mol is None:
            continue
        
        matching_centers = []
        for idx, (sm_center, center_mol) in enumerate(center_mols):
            if center_mol is None:
                continue
            # 检查反应类型是否匹配
            if rxn_type not in smarts_type_set.get(sm_center, set()):
                continue
            # 子结构匹配
            if mol.HasSubstructMatch(center_mol):
                matching_centers.append(idx)
        
        if matching_centers:
            prod_center_maps[(rxn_type, cano_prod)] = matching_centers
    
    return prod_center_maps


# 执行 Step 4
prod_center_maps = find_product_centers(
    MINI_REACTIONS, smiles_cano_map, prod_cano_smarts, smarts_cano_map, filtered_templates
)

print(f'Step 4 完成: 反应中心匹配')
print(f'  有效产物-中心映射数: {len(prod_center_maps)}')
print()
print('产物 → 匹配中心映射表:')
print('-' * 70)
for (rxn_type, prod), centers in prod_center_maps.items():
    center_smarts = [prod_cano_smarts[c] for c in centers]
    print(f'  类型 {rxn_type} | 产物: {prod}')
    print(f'          匹配中心索引: {centers}')
    print(f'          匹配中心SMARTS: {center_smarts}')
    print()

Step 4 完成: 反应中心匹配
  有效产物-中心映射数: 966

产物 → 匹配中心映射表:
----------------------------------------------------------------------
  类型 nan | 产物: Cc1nc2c(s1)C(=O)/C(=C\N(C)C)CC2
          匹配中心索引: [223]
          匹配中心SMARTS: ['C/[C&H0&D3&+0](=[C&H1&D2&+0]/[N&H0&D3&+0](-[C&D1&H3])-[C&D1&H3])-C=[O&D1&H0]']

  类型 nan | 产物: O=C(NC1C=CC(O)CC1)OCc1ccccc1
          匹配中心索引: [564]
          匹配中心SMARTS: ['[#8]-[C&H0&D3&+0](=[O&D1&H0])-[N&H1&D2&+0]-[C&H1&D3&+0](-C)-C=C']

  类型 nan | 产物: Cc1ccc(N=C/C=C/N(C(=O)c2ccccc2)c2ccc(C)cc2)cc1
          匹配中心索引: [646]
          匹配中心SMARTS: ['[C&H3&D1&+0]-[c&H0&D3&+0]1:c:[c&H1&D2&+0]:[c&H0&D3&+0](-[N&H0&D2&+0]=[C&H1&D2&+0]/[C&H1&D2&+0]=[C&H1&D2&+0]/[N&H0&D3&+0](-[C&H0&D3&+0](=[O&D1&H0])-c)-[c&H0&D3&+0]2:c:c:[c&H0&D3&+0](-[C&H3&D1&+0]):c:c:2):c:c:1.c:[c&H1&D2&+0]:c']

  类型 nan | 产物: COc1ccc2c(c1)OC[C@@H]1CCCN1C2=O
          匹配中心索引: [621]
          匹配中心SMARTS: ['[C&D1&H3]-[O&H0&D2&+0]-[c&H0&D3&+0](:c):c']

  类型 nan | 产物: CC(=O)N[C@@H](CCCCNC(=O)OCc1ccccc1)C(=O)N[C@H](Cc1

---

## Step 5: 构建正负样本

### 训练数据的构建逻辑

GLN 的训练需要**对比学习**式的正负样本：

1. **正样本**: 给定产物，真实的模板和反应物
2. **负样本**: 
   - **负中心**: 能匹配产物但不是真实反应中心的 SMARTS
   - **负模板**: 同一中心下的其他模板（不产生正确反应物）
   - **负反应物**: 模板应用后产生的错误反应物

具体流程：
```
对于每个训练反应 (产物p, 正确反应物r):
  1. 查找产物的所有匹配中心
  2. 对每个中心，找到关联的模板
  3. 对每个模板，应用到产物上生成预测反应物
  4. 预测反应物 == 正确反应物 → 正样本模板
  5. 预测反应物 != 正确反应物 → 负样本反应物
```

### 模板应用

GLN 使用 `Reactor` 类（封装了 rdChiral）将模板逆向应用到产物上，得到预测的反应物。

### GLN 源码对应
- 文件: `gln/data_process/build_all_reactions.py`
- 类: `gln/common/reactor.py → Reactor`

In [8]:
# ====== Step 5: 正负样本构建 ======
# 对应 GLN 源码: gln/data_process/build_all_reactions.py

# --- 简化版 Reactor ---
# GLN 使用 rdChiral 来应用模板。这里展示其核心逻辑。

try:
    from rdchiral.main import rdchiralReaction, rdchiralReactants, rdchiralRun
    HAS_RDCHIRAL = True
except ImportError:
    HAS_RDCHIRAL = False


class SimpleReactor:
    """
    模板应用器。
    将逆合成模板应用到产物上，生成预测的反应物。
    
    对应 GLN 源码: gln/common/reactor.py → _Reactor
    GLN 使用缓存来避免重复计算。
    """
    def __init__(self):
        self.rxn_cache = {}    # 缓存编译后的模板
        self.src_cache = {}    # 缓存编译后的产物
        self.result_cache = {} # 缓存反应结果
    
    def run_reaction(self, product_smiles, template):
        """
        将模板应用到产物上。
        
        参数:
            product_smiles: 产物SMILES
            template: 逆合成模板 (prod_smarts>agent>react_smarts)
        返回:
            可能的反应物SMILES列表
        """
        key = (product_smiles, template)
        if key in self.result_cache:
            return self.result_cache[key]
        
        if not HAS_RDCHIRAL:
            # 无rdchiral时使用RDKit的RunReactants
            return self._rdkit_run(product_smiles, template)
        
        try:
            # 确保模板格式正确
            p, a, r = template.split('>')
            if '.' in p and p[0] != '(':
                p = '(' + p + ')'
            tpl = '>'.join((p, a, r))
            
            rxn = rdchiralReaction(tpl)
            src = rdchiralReactants(product_smiles)
            outcomes = rdchiralRun(rxn, src)
            self.result_cache[key] = outcomes
            return outcomes
        except:
            self.result_cache[key] = None
            return None
    
    def _rdkit_run(self, product_smiles, template):
        """使用 RDKit 原生反应引擎作为后备"""
        try:
            # 将模板转换为反应SMARTS
            parts = template.split('>')
            if len(parts) == 3:
                rxn_smarts = f'{parts[0]}>>{parts[2]}'
            else:
                return None
            rxn = AllChem.ReactionFromSmarts(rxn_smarts)
            if rxn is None:
                return None
            mol = Chem.MolFromSmiles(product_smiles)
            if mol is None:
                return None
            results = rxn.RunReactants([mol])
            outcomes = []
            for products in results:
                smis = '.'.join(sorted([Chem.MolToSmiles(p) for p in products]))
                if smis not in outcomes:
                    outcomes.append(smis)
            self.result_cache[(product_smiles, template)] = outcomes
            return outcomes
        except:
            return None


reactor = SimpleReactor()
print('SimpleReactor 初始化完成')

SimpleReactor 初始化完成


In [9]:
# ====== 演示模板应用 ======

print('模板应用演示（将逆合成模板应用到产物上）')
print('=' * 70)

for rxn_type, tpl in filtered_templates:
    print(f'\n模板 (类型 {rxn_type}): {tpl}')
    parts = tpl.split('>')
    if len(parts) != 3:
        print('  [非标准格式，跳过]')
        continue
    
    # 找一个匹配该模板的产物来演示
    for _, rxn in MINI_REACTIONS:
        _, _, raw_prod = rxn.split('>')
        cano_prod = smiles_cano_map.get(raw_prod, raw_prod)
        
        result = reactor.run_reaction(cano_prod, tpl)
        if result and len(result) > 0:
            print(f'  产物: {cano_prod}')
            print(f'  预测反应物: {result}')
            break

模板应用演示（将逆合成模板应用到产物上）

模板 (类型 nan): [C:3]/[C;H0;D3;+0:2](=[CH;D2;+0:1]/[N;H0;D3;+0:7](-[C;D1;H3:6])-[C;D1;H3:8])-[C:4]=[O;D1;H0:5]>>O=[CH;D2;+0:1]-[CH;D3;+0:2](-[C:3])-[C:4]=[O;D1;H0:5].[C;D1;H3:6]-[NH;D2;+0:7]-[C;D1;H3:8]
  产物: Cc1nc2c(s1)C(=O)/C(=C\N(C)C)CC2
  预测反应物: ['CNC.Cc1nc2c(s1)C(=O)C(C=O)CC2']

模板 (类型 nan): [#8:2]-[C;H0;D3;+0:1](=[O;D1;H0:3])-[NH;D2;+0:6]-[CH;D3;+0:5](-[C:4])-[C:7]=[C:8]>>O=C1-C-C-C(=O)-N-1-O-[C;H0;D3;+0:1](-[#8:2])=[O;D1;H0:3].[C:4]-[C@@H;D3;+0:5](-[NH2;D1;+0:6])-[C:7]=[C:8]
  产物: O=C(NC1C=CC(O)CC1)OCc1ccccc1
  预测反应物: ['N[C@H]1C=CC(O)CC1.O=C(OCc1ccccc1)ON1C(=O)CCC1=O']

模板 (类型 nan): [CH3;D1;+0:4]-[c;H0;D3;+0:3]1:[c:2]:[cH;D2;+0:1]:[c;H0;D3;+0:7](-[N;H0;D2;+0:25]=[CH;D2;+0:21]/[CH;D2;+0:22]=[CH;D2;+0:24]/[N;H0;D3;+0:23](-[C;H0;D3;+0:8](=[O;D1;H0:9])-[c:10])-[c;H0;D3;+0:15]2:[c:16]:[c:17]:[c;H0;D3;+0:18](-[CH3;D1;+0:14]):[c:19]:[c:20]:2):[c:6]:[c:5]:1.[c:12]:[cH;D2;+0:11]:[c:13]>>C-c1:c(-C-C(-Cl)=O):[c;H0;D3;+0:1]2:[c:2]:[c;H0;D3;+0:3](-O-[CH3;D1;+0:4]):[c:5]:[c

In [11]:
# ====== 构建正负样本 ======

def build_training_samples(reactions, smiles_cano_map, prod_center_maps,
                           prod_cano_smarts, smarts_cano_map,
                           unique_templates, reactor):
    """
    构建训练所需的正负样本。
    
    对应 GLN 源码: build_all_reactions.py → find_tpls()
    
    返回:
        samples: 列表，每项包含 (产物, 正确模板索引, 负样本反应物)
    """
    # 构建中心 → 模板的映射
    tpl_of_center = defaultdict(lambda: defaultdict(list))
    for i, (rxn_type, tpl) in enumerate(unique_templates):
        parts = tpl.split('>')
        if len(parts) != 3:
            continue
        center = parts[0]
        if smarts_has_useless_parentheses(center):
            center = center[1:-1]
        if center in smarts_cano_map:
            cano_center = smarts_cano_map[center]
            tpl_of_center[cano_center][rxn_type].append(i)
    
    samples = []
    for idx, (rxn_type, rxn) in enumerate(reactions):
        reactants_raw, _, raw_prod = rxn.split('>')
        cano_prod = smiles_cano_map.get(raw_prod, raw_prod)
        cano_reactants = smiles_cano_map.get(reactants_raw, reactants_raw)
        
        if (rxn_type, cano_prod) not in prod_center_maps:
            continue
        
        center_indices = prod_center_maps[(rxn_type, cano_prod)]
        
        pos_tpl_indices = []
        neg_reactants = set()
        
        for center_idx in center_indices:
            center_smarts = prod_cano_smarts[center_idx]
            if center_smarts not in tpl_of_center:
                continue
            if rxn_type not in tpl_of_center[center_smarts]:
                continue
            
            tpl_indices = tpl_of_center[center_smarts][rxn_type]
            
            for tpl_idx in tpl_indices:
                _, tpl = unique_templates[tpl_idx]
                # 应用模板到产物
                pred_mols = reactor.run_reaction(cano_prod, tpl)
                if pred_mols is None or len(pred_mols) == 0:
                    continue
                for pred in pred_mols:
                    if pred == cano_reactants:
                        pos_tpl_indices.append(tpl_idx)
                    else:
                        neg_reactants.add(pred)
        
        samples.append({
            'idx': idx,
            'product': cano_prod,
            'true_reactants': cano_reactants,
            'rxn_type': rxn_type,
            'pos_templates': pos_tpl_indices,
            'neg_reactants': list(neg_reactants)
        })
    
    return samples


# 将 filtered_templates 排序形成唯一模板列表（与 GLN 的 DataInfo.unique_templates 对应）
unique_templates = sorted(filtered_templates)

training_samples = build_training_samples(
    MINI_REACTIONS, smiles_cano_map, prod_center_maps,
    prod_cano_smarts, smarts_cano_map, unique_templates, reactor
)

print(f'Step 5 完成: 正负样本构建')
print(f'  有效训练样本数: {len(training_samples)}')
print()
for sample in training_samples:
    print(f'反应 {sample["idx"]+1} (类型 {sample["rxn_type"]}):')
    print(f'  产物:       {sample["product"]}')
    print(f'  正确反应物: {sample["true_reactants"]}')
    print(f'  正模板索引: {sample["pos_templates"]}')
    print(f'  负反应物数: {len(sample["neg_reactants"])}')
    if sample['neg_reactants']:
        print(f'  负反应物:   {sample["neg_reactants"][:3]}...')
    print()

Step 5 完成: 正负样本构建
  有效训练样本数: 966

反应 1 (类型 nan):
  产物:       Cc1nc2c(s1)C(=O)/C(=C\N(C)C)CC2
  正确反应物: C1CCOC1.CCOCC.Cc1ccccc1.O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[c:4]([n:3][c:2]([CH3:1])[s:6]2)[CH2:15][CH2:14]1.[NH:11]([CH3:12])[CH3:13]
  正模板索引: []
  负反应物数: 1
  负反应物:   ['CNC.Cc1nc2c(s1)C(=O)C(C=O)CC2']...

反应 2 (类型 nan):
  产物:       O=C(NC1C=CC(O)CC1)OCc1ccccc1
  正确反应物: CCN(CC)CC.CCOCC.CN(C)C=O.O=C1CCC(=O)N1O[C:2](=[O:1])[O:11][CH2:12][c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1.[NH2:3][C@H:4]1[CH:5]=[CH:6][C@@H:7]([OH:8])[CH2:9][CH2:10]1
  正模板索引: []
  负反应物数: 1
  负反应物:   ['N[C@H]1C=CC(O)CC1.O=C(OCc1ccccc1)ON1C(=O)CCC1=O']...

反应 3 (类型 nan):
  产物:       Cc1ccc(N=C/C=C/N(C(=O)c2ccccc2)c2ccc(C)cc2)cc1
  正确反应物: Cc1c(CC(=O)Cl)[c:26]2[c:5](n1[C:11](=[O:12])[c:13]1[cH:14][cH:15][c:16](Cl)[cH:17][cH:18]1)[cH:4][cH:3][c:2](O[CH3:1])[cH:27]2.Cc1c(CC(=O)O)c2cc(O[CH3:23])ccc2n1C(=O)[c:19]1[cH:20][cH:21][c:22](Cl)[cH:24][cH:25]1.Cl.ClCCl.O=S(Cl)Cl.[nH:6]1[cH:7][cH:8][n:10][cH:9]1
  正模板索引: []
  负反应物数

---

## Step 6: 分子图特征导出

### 将分子转换为图结构

GLN 将每个分子/子结构表示为**图（Graph）**：
- **节点 = 原子**，特征包括：原子序数、度数、氢原子数、化合价、芳香性
- **边 = 化学键**，特征包括：键类型（单/双/三/芳香）、共轭性、是否在环内

GLN 用位打包（bit packing）将这些离散特征编码为单个整数，以节省内存：

```
原子特征 (32-bit 整数):
┌──────────┬────────────┬─────────┬────────┬──────────┐
│ 芳香性(1) │ 隐式化合价(4) │ 氢计数(4) │ 度数(4) │ 原子序数(8) │
└──────────┴────────────┴─────────┴────────┴──────────┘

键特征 (32-bit 整数):
┌──────────┬──────────┬──────────┐
│ 环内(1+7) │ 共轭(1+7) │ 键类型(8) │
└──────────┴──────────┴──────────┘
```

### GLN 源码对应
- 文件: `gln/data_process/dump_graphs.py`, `gln/mods/mol_gnn/mol_utils.py`
- 类: `MolGraph`, `SmilesMols`, `SmartsMols`

In [18]:
# ====== Step 6: 分子图特征提取 ======
# 对应 GLN 源码: gln/mods/mol_gnn/mol_utils.py

import rdkit
from rdkit import Chem

# GLN 使用的原子类型列表（简化版）
ATOM_LIST = [6, 7, 8, 9, 15, 16, 17, 35, 53]  # C, N, O, F, P, S, Cl, Br, I
ATOM_IDX_MAP = {a: i for i, a in enumerate(ATOM_LIST)}


def get_atom_feat(atom, sanitized=True):
    """
    提取单个原子的特征，编码为一个 32-bit 整数。
    
    对应 GLN 源码: mol_utils.py → get_atom_feat()
    
    特征编码（从高位到低位）:
    - bit[20]: 芳香性 (0/1)
    - bit[16:19]: 隐式化合价 (0-6)
    - bit[12:15]: 氢原子总数 (0-5)
    - bit[8:11]: 度数 (连接数)
    - bit[0:7]: 原子类型索引
    
    注意: 对于 SMARTS 分子 (sanitized=False)，RDKit 2025+ 不再允许直接
    调用 GetImplicitValence() / GetTotalNumHs()，因为查询分子未计算化合价。
    GLN 源码中对 GetTotalNumHs 已做特殊处理 (hardcoded=5)，
    这里对 GetImplicitValence 也做同样处理 (hardcoded=6)。
    """
    # 芳香性
    feat = int(atom.GetIsAromatic())
    feat <<= 4
    
    # 隐式化合价
    # SMARTS 查询分子在新版 RDKit 中无法调用 GetImplicitValence()
    # GLN 源码在旧版 RDKit 上可直接调用，但新版需要特殊处理
    if sanitized:
        v = atom.GetImplicitValence()
        feat |= min(v, 6)
    else:
        feat |= 6  # SMARTS 模式下使用溢出值（表示"未知"）
    feat <<= 4
    
    # 氢原子总数
    if sanitized:
        h = atom.GetTotalNumHs()
        feat |= min(h, 4)
    else:
        feat |= 5  # SMARTS 模式下的特殊值
    feat <<= 4
    
    # 度数
    feat |= atom.GetDegree()
    feat <<= 8
    
    # 原子类型
    atomic_num = atom.GetAtomicNum()
    if atomic_num in ATOM_IDX_MAP:
        feat |= ATOM_IDX_MAP[atomic_num]
    else:
        feat |= len(ATOM_IDX_MAP)  # 未知原子类型
    
    return feat


def get_bond_feat(bond, sanitized=True):
    """
    提取单个化学键的特征，编码为一个整数。
    
    对应 GLN 源码: mol_utils.py → get_bond_feat()
    """
    bt = bond.GetBondType()
    t = 0
    if bt == rdkit.Chem.rdchem.BondType.SINGLE:
        t = 0
    elif bt == rdkit.Chem.rdchem.BondType.DOUBLE:
        t = 1
    elif bt == rdkit.Chem.rdchem.BondType.TRIPLE:
        t = 2
    elif bt == rdkit.Chem.rdchem.BondType.AROMATIC:
        t = 3
    
    # 是否在环内
    feat = 2  # 默认值（用于 SMARTS 模式）
    if sanitized:
        feat = 1 if bond.GetOwningMol().GetRingInfo().NumBondRings(bond.GetIdx()) > 0 else 0
    feat <<= 8
    
    # 共轭性
    feat |= int(bond.GetIsConjugated())
    feat <<= 8
    
    # 键类型
    feat |= t
    
    return feat


class MolGraph:
    """
    分子图数据结构。
    
    对应 GLN 源码: mol_utils.py → MolGraph
    
    属性:
        name: 分子的SMILES/SMARTS名称
        num_nodes: 原子数量
        num_edges: 化学键数量
        atom_feats: 原子特征数组 (int32)
        bond_feats: 键特征数组 (int32)
        edge_pairs: 边的起止节点对 [..., src, dst, ...]
    """
    def __init__(self, name, mol, sanitized=True):
        self.name = name
        self.sanitized = sanitized
        self.num_nodes = mol.GetNumAtoms()
        self.num_edges = mol.GetNumBonds()
        
        # 提取原子特征
        self.atom_feats = np.zeros(self.num_nodes, dtype=np.int32)
        for i, atom in enumerate(mol.GetAtoms()):
            self.atom_feats[i] = get_atom_feat(atom, sanitized)
        
        # 提取键特征
        self.bond_feats = np.zeros(self.num_edges, dtype=np.int32)
        self.edge_pairs = np.zeros(self.num_edges * 2, dtype=np.int32)
        for i, bond in enumerate(mol.GetBonds()):
            self.bond_feats[i] = get_bond_feat(bond, sanitized)
            self.edge_pairs[i * 2] = bond.GetBeginAtomIdx()
            self.edge_pairs[i * 2 + 1] = bond.GetEndAtomIdx()


# ====== 演示分子图构建 ======
demo_smiles = 'CC(=O)OC'  # 乙酸甲酯
mol = Chem.MolFromSmiles(demo_smiles)
mg = MolGraph(demo_smiles, mol, sanitized=True)

print(f'分子图演示: {demo_smiles}')
print(f'  节点数 (原子): {mg.num_nodes}')
print(f'  边数 (化学键): {mg.num_edges}')
print(f'  原子特征 (打包整数): {mg.atom_feats}')
print(f'  键特征 (打包整数):   {mg.bond_feats}')
print(f'  边连接对: {mg.edge_pairs.reshape(-1, 2).tolist()}')

分子图演示: CC(=O)OC
  节点数 (原子): 5
  边数 (化学键): 4
  原子特征 (打包整数): [209152    768    258    514 209152]
  键特征 (打包整数):   [  0 257 256   0]
  边连接对: [[0, 1], [1, 2], [1, 3], [3, 4]]


In [19]:
# ====== 解码特征查看（教学目的）======

def decode_atom_feat(feat):
    """将打包的原子特征解码为可读形式"""
    atomic_idx = feat & 0xFF
    feat >>= 8
    degree = feat & 0xF
    feat >>= 4
    h_count = feat & 0xF
    feat >>= 4
    valence = feat & 0xF
    feat >>= 4
    aromatic = feat & 0x1
    
    # 反查原子类型
    idx_atom_map = {v: k for k, v in ATOM_IDX_MAP.items()}
    atomic_num = idx_atom_map.get(atomic_idx, '?')
    
    return {
        'atomic_num': atomic_num,
        'degree': degree,
        'h_count': h_count,
        'valence': valence,
        'aromatic': bool(aromatic)
    }


def decode_bond_feat(feat):
    """将打包的键特征解码为可读形式"""
    bond_type_map = {0: 'SINGLE', 1: 'DOUBLE', 2: 'TRIPLE', 3: 'AROMATIC'}
    bond_type = feat & 0xFF
    feat >>= 8
    conjugated = bool(feat & 0xFF)
    feat >>= 8
    in_ring = feat & 0xFF
    
    return {
        'bond_type': bond_type_map.get(bond_type, '?'),
        'conjugated': conjugated,
        'in_ring': in_ring == 1
    }


print(f'分子 {demo_smiles} 的解码特征:')
print()
print('原子特征:')
print(f'{"索引":>4} {"符号":>4} {"原子序数":>8} {"度数":>4} {"氢计数":>6} {"化合价":>6} {"芳香":>6}')
print('-' * 50)
for i, (atom, feat) in enumerate(zip(mol.GetAtoms(), mg.atom_feats)):
    decoded = decode_atom_feat(feat)
    print(f'{i:4d} {atom.GetSymbol():>4} {decoded["atomic_num"]:>8} '
          f'{decoded["degree"]:4d} {decoded["h_count"]:6d} '
          f'{decoded["valence"]:6d} {str(decoded["aromatic"]):>6}')

print()
print('键特征:')
print(f'{"索引":>4} {"起点":>4} {"终点":>4} {"键类型":>10} {"共轭":>8} {"环内":>6}')
print('-' * 50)
for i in range(mg.num_edges):
    decoded = decode_bond_feat(mg.bond_feats[i])
    src = mg.edge_pairs[i * 2]
    dst = mg.edge_pairs[i * 2 + 1]
    print(f'{i:4d} {src:4d} {dst:4d} {decoded["bond_type"]:>10} '
          f'{str(decoded["conjugated"]):>8} {str(decoded["in_ring"]):>6}')

分子 CC(=O)OC 的解码特征:

原子特征:
  索引   符号     原子序数   度数    氢计数    化合价     芳香
--------------------------------------------------
   0    C        6    1      3      3  False
   1    C        6    3      0      0  False
   2    O        8    1      0      0  False
   3    O        8    2      0      0  False
   4    C        6    1      3      3  False

键特征:
  索引   起点   终点        键类型       共轭     环内
--------------------------------------------------
   0    0    1     SINGLE    False  False
   1    1    2     DOUBLE     True  False
   2    1    3     SINGLE     True  False
   3    3    4     SINGLE    False  False


In [20]:
# ====== 批量构建分子图 ======
# GLN 的 SmilesMols / SmartsMols 就是这样的分子图缓存池

class MolGraphPool:
    """
    分子图缓存池。
    对应 GLN 源码: mol_utils.py → _MolHolder / SmilesMols / SmartsMols
    
    GLN 分别维护两个池:
    - SmilesMols: 处理 SMILES (sanitized=True)
    - SmartsMols: 处理 SMARTS (sanitized=False)
    """
    def __init__(self, sanitized=True):
        self.sanitized = sanitized
        self.cache = {}
    
    def get_mol_graph(self, name):
        if name is None or len(name.strip()) == 0:
            return None
        if name not in self.cache:
            if self.sanitized:
                mol = Chem.MolFromSmiles(name)
            else:
                mol = Chem.MolFromSmarts(name)
            if mol is None:
                return None
            self.cache[name] = MolGraph(name, mol, self.sanitized)
        return self.cache[name]


# 为所有出现的SMILES/SMARTS构建图
smiles_pool = MolGraphPool(sanitized=True)
smarts_pool = MolGraphPool(sanitized=False)

# 构建SMILES图
for sm in set(smiles_cano_map.values()):
    smiles_pool.get_mol_graph(sm)

# 构建SMARTS图
for sm in prod_cano_smarts + react_cano_smarts:
    smarts_pool.get_mol_graph(sm)

print(f'SMILES 图缓存: {len(smiles_pool.cache)} 个分子')
print(f'SMARTS 图缓存: {len(smarts_pool.cache)} 个子结构模式')

SMILES 图缓存: 2902 个分子
SMARTS 图缓存: 1616 个子结构模式


---

## 完整流程总结

### 数据处理管线回顾

| Step | 任务 | 输入 | 输出 | GLN 源文件 |
|------|------|------|------|------------|
| 1 | SMILES 标准化 | 原始反应 CSV | `cano_smiles.pkl` + `atom_list.txt` | `get_canonical_smiles.py` |
| 2 | 模板提取 | 训练集反应 | `proc_train_singleprod.csv` | `build_raw_template.py` |
| 3a | 模板过滤 | 提取的模板 | `templates.csv` | `filter_template.py` |
| 3b | SMARTS标准化 | 模板列表 | `cano_smarts.pkl` + `prod/react_cano_smarts.txt` | `get_canonical_smarts.py` |
| 4 | 反应中心匹配 | 产物+中心 SMARTS | `prod_center_maps.csv` | `find_centers.py` |
| 5 | 正负样本构建 | 产物+模板+Reactor | `pos_tpls.csv` + `neg_reacts.csv` | `build_all_reactions.py` |
| 6 | 图特征导出 | 分子/子结构 | `.bin` + `.names` 二进制文件 | `dump_graphs.py` |

### 关键数据结构

| 数据结构 | 说明 | 对应 GLN 代码 |
|----------|------|---------------|
| `smiles_cano_map` | 原始SMILES → 标准化SMILES | `DataInfo.smiles_cano_map` |
| `smarts_cano_map` | 原始SMARTS → 标准化SMARTS | `DataInfo.smarts_cano_map` |
| `prod_cano_smarts` | 所有唯一产物中心列表 | `DataInfo.prod_cano_smarts` |
| `prod_center_maps` | (类型, 产物) → 中心索引列表 | `DataInfo.prod_center_maps` |
| `unique_templates` | 排序后的 (类型, 模板) 列表 | `DataInfo.unique_templates` |
| `tpl_of_center` | 中心 → 类型 → 模板索引列表 | `DataInfo.unique_tpl_of_prod_center` |
| `MolGraph` | 分子的图表示 | `mol_utils.MolGraph` |
| `SmilesMols/SmartsMols` | 分子图缓存池 | `mol_utils._MolHolder` |

In [21]:
# ====== 可视化数据处理流程 ======

print('\n' + '='*70)
print('GLN 数据处理流程完成！')
print('='*70)
print(f'''
统计汇总:
  原始反应数:         {len(MINI_REACTIONS)}
  唯一SMILES:         {len(set(smiles_cano_map.values()))}
  提取模板数:         {len(valid_templates)}
  唯一产物中心:       {len(prod_cano_smarts)}
  唯一反应物中心:     {len(react_cano_smarts)}
  产物-中心映射:      {len(prod_center_maps)}
  训练样本数:         {len(training_samples)}
  SMILES图缓存:       {len(smiles_pool.cache)}
  SMARTS图缓存:       {len(smarts_pool.cache)}
''')


GLN 数据处理流程完成！

统计汇总:
  原始反应数:         1000
  唯一SMILES:         2902
  提取模板数:         966
  唯一产物中心:       726
  唯一反应物中心:     915
  产物-中心映射:      966
  训练样本数:         966
  SMILES图缓存:       2902
  SMARTS图缓存:       1616



## 补充：与实际数据集的对比

本教程使用的微型数据集只有 5 条反应。实际 GLN 论文使用的 **Schneider50k** 数据集：

| 统计项 | 微型数据集 | Schneider50k |
|--------|-----------|-------------|
| 训练反应数 | 5 | ~40,000 |
| 唯一模板数 | ~3 | ~11,000 |
| 反应类型数 | 2 | 50 |
| 产物中心数 | ~2 | ~4,000 |
| 处理时间 | 秒级 | 小时级（需多核并行） |

在实际使用中，GLN 用 **多进程并行**（`multiprocessing.Pool`）加速 Step 2/4/5，并将数据分块处理（`num_parts` 参数）以支持分布式。

**下一步** → 请打开 `3_模型展示.ipynb`，学习 GLN 的模型架构和推理流程。